# 02 - Full Pipeline Loop

Runs the complete pipeline on ALL images in the dataset.
Builds `F_dataset.npy` and `labels.npy` for FANN classification.

Pipeline per image:
1. Preprocessing       → pb
2. SCH-CS              → schcs_binary, n_sr
3. CHM correction      → p_bar
4. φ initialization    → phi_init
5. DLPE iteration      → segmented_sr
6. SR split            → sr_left, sr_right
7. Feature extraction  → f_v_left, f_v_right
8. Asymmetry vector    → F (21,)

Then: FANN classification on full F_dataset (N, 21)

## Imports

In [1]:
import cv2
import sys, os
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.append(os.path.abspath(".."))

# ── Your existing src modules ──────────────────────────────────────────────
from src.sr_segmentation.main import (
run_schcs,
run_sr_split,
run_chm_correction,
run_phi_initialization,
run_level_set_iteration
)
from src.preprocessing import run_preprocessing
from sklearn.preprocessing import StandardScaler
from src.asymmetry_vector import run_asymmetry_pipeline
from src.feature_extraction import run_feature_extraction
from src.utils import section, PRE_CFG, SCH_CFG, base_path, dmr_ir_o
from src.fann_classifier import run_cross_validation, visualize_results, build_fann

print('All imports OK')

ImportError: cannot import name 'run_schcs' from 'src.sr_segmentation.main' (D:\Projects\Dissertation\src\sr_segmentation\main.py)

## Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DMR_IR_PATH    = base_path + dmr_ir_o
RESULTS_DIR    = os.path.join('..', 'results', 'full_pipeline')
DATASET_DIR    = os.path.join('..', 'results', 'dataset')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

# ── DLPE parameters (tuned from single-image experiments) ─────────────────
DLPE_CFG = dict(
    alpha1         = 1.0,
    alpha2         = 1.0,
    theta          = 0.2,
    epsilon        = 1.5,
    dt             = 0.01,    # reduced from 0.1
    t_stop         = 0.001,   # tightened from 0.05
    max_iterations = 1000,
    verbose_every  = 999      # suppress per-iteration prints in loop
)

# ── CHM window (fixed from single-image experiments) ──────────────────────
CHM_WINDOW = 51

# ── Label map: image_name → 0 (Normal) or 1 (Abnormal) ───────────────────
# Option A: Load from CSV if you have one
# df = pd.read_csv(os.path.join(base_path, 'labels.csv'))
# LABEL_MAP = dict(zip(df['image_name'], df['label']))

# Option B: Define manually (DMR-IR database ground truth)
# 0 = Normal, 1 = Abnormal
# Fill this in based on your DMR-IR database documentation
LABEL_MAP = {
    'TFRON_V11_31-10-2012_0'  : 0,
    'TFRON_V12_31-10-2012_0'  : 0,
    'TFRON_V13_31-10-2012_0'  : 0,
    'TFRON_V15_31-10-2012_0'  : 0,
    'TFRON_V16_31-10-2012_0'  : 0,
    'TFRON_V1_26-10-2012_0'   : 0,
    'TFRON_V243_12-3-2014_0'  : 1,   # ← the image you've been working with
    'TFRON_V247_21-5-2014_0'  : 1,
    'TFRON_V2_30-10-2012_0'   : 0,
    'TFRON_V39_7-11-2012_0'   : 1,
    'TFRON_V4_30-10-2012_0'   : 0,
    'TFRON_V59_20-11-2012_0'  : 1,
    'TFRON_V5_30-10-2012_0'   : 0,
    'TFRON_V6_30-10-2012_0'   : 0,
    'TFRON_V7_31-10-2012_0'   : 0,
    'TFRON_V8_31-10-2012_0'   : 0,
    'TFRON_V9_31-10-2012_0'   : 0,
    # ↑ UPDATE THESE with correct labels from DMR-IR documentation
}

print(f'Results dir : {RESULTS_DIR}')
print(f'Dataset dir : {DATASET_DIR}')
print(f'Label map   : {len(LABEL_MAP)} entries')

## Full Pipeline Loop

In [ ]:
# ── Collect all image files ────────────────────────────────────────────────
image_files = sorted([
    f for f in os.listdir(DMR_IR_PATH)
    if f.lower().endswith(('.jpg', '.png', '.jpeg'))
])
print(f'Found {len(image_files)} images to process')

# ── Dataset accumulators ───────────────────────────────────────────────────
all_F      = []   # will become F_dataset (N, 21)
all_labels = []   # will become labels    (N,)
failed     = []   # images that errored — (name, reason)
skipped    = []   # images with no label

# ── Main loop ─────────────────────────────────────────────────────────────
for idx, fname in enumerate(image_files):

    image_name = Path(fname).stem
    image_path = os.path.join(DMR_IR_PATH, fname)

    print(f'\n{"-"*60}')
    print(f'[{idx+1}/{len(image_files)}] {image_name}')
    print(f'{"-"*60}')

    # ── Check label exists before doing any work ───────────────────────────
    if image_name not in LABEL_MAP:
        print(f'  [SKIP] No label found in LABEL_MAP')
        skipped.append(image_name)
        continue

    label = LABEL_MAP[image_name]

    try:
        # ── Per-image output folder ────────────────────────────────────────
        img_out = os.path.join(RESULTS_DIR, image_name)
        os.makedirs(img_out, exist_ok=True)

        # ── STEP 1: Preprocessing ──────────────────────────────────────────
        print('  [1/8] Preprocessing...')
        preprocessing_results = run_preprocessing(
            image_path     = image_path,
            config         = PRE_CFG,
            storage_config = SCH_CFG,
            show_visualization = False    # suppress plots in loop
        )
        pb         = preprocessing_results['pb']
        image_name = preprocessing_results['image_name']

        # ── STEP 2: SCH-CS ─────────────────────────────────────────────────
        print('  [2/8] SCH-CS...')
        schcs_results = run_schcs(
            pb         = pb,
            image_name = image_name,
            show_visualization = False
        )
        schcs_binary = schcs_results['binary']
        n_sr         = schcs_results['n_sr']
        print(f'         n_sr={n_sr}, coverage={schcs_binary.mean()*100:.1f}%')

        # ── STEP 3: CHM correction ─────────────────────────────────────────
        print('  [3/8] CHM correction (window=51)...')
        p_bar = compute_p_bar_thermal(pb, window_size=CHM_WINDOW)

        # Quick contrast check — warn if too low
        from src.sr_segmentation.dlpe_iteration_loop import update_region_means
        phi_tmp = 4.0*schcs_binary.astype(np.float64) - (1-schcs_binary.astype(np.float64))*4.0
        l1_check, l2_check = update_region_means(p_bar, phi_tmp)
        contrast = abs(l1_check - l2_check)
        print(f'         |l1-l2|={contrast:.2f}', end='')
        if contrast < 20:
            print(' [WARN: low contrast, results may be noisy]')
        else:
            print(' [OK]')

        # ── STEP 4: φ initialization ───────────────────────────────────────
        print('  [4/8] φ initialization...')
        phi_init_results = run_phi_initialization(
            schcs_binary = schcs_binary,
            image_name   = image_name,
            show_visualization = False
        )
        phi_init = phi_init_results['phi']

        # ── STEP 5: DLPE iteration ─────────────────────────────────────────
        print('  [5/8] DLPE level set...')
        level_set_results = run_level_set_iteration(
            p_bar      = p_bar,
            phi_init   = phi_init,
            n_sr       = n_sr,
            image_name = image_name,
            show_visualization = False,
            **DLPE_CFG
        )
        segmented_sr = level_set_results['segmented_sr']
        iters        = level_set_results.get('iterations', '?')
        coverage     = segmented_sr.mean()*100
        print(f'         iters={iters}, coverage={coverage:.1f}%')

        # ── STEP 6: Post-process + split SR ───────────────────────────────
        print('  [6/8] SR post-process + split...')
        seg_filtered = extract_breast_srs(segmented_sr, pb)
        sr_left, sr_right, centre_col = split_sr_left_right(
            seg_filtered, pb
        )
        print(f'         left={sr_left.sum()}px, right={sr_right.sum()}px, '
              f'centre={centre_col}')

        # Warn if one side is empty
        if sr_left.sum() == 0 or sr_right.sum() == 0:
            print(f'         [WARN] One breast has no SR — asymmetry may be unreliable')

        # ── STEP 7: Feature extraction ─────────────────────────────────────
        print('  [7/8] Feature extraction...')
        feature_results = run_feature_extraction(
            pb         = pb,
            sr_left    = sr_left,
            sr_right   = sr_right,
            image_name = image_name,
            show_visualization = False
        )
        f_v_left  = feature_results['f_v_left']
        f_v_right = feature_results['f_v_right']

        # ── STEP 8: Asymmetry vector ───────────────────────────────────────
        print('  [8/8] Asymmetry vector...')
        asymmetry_results = run_asymmetry_pipeline(
            f_v_left   = f_v_left,
            f_v_right  = f_v_right,
            image_name = image_name,
            show_visualization = False
        )
        F = asymmetry_results['F']   # shape (21,)

        # ── Save per-image F and metadata ──────────────────────────────────
        np.save(os.path.join(img_out, 'F.npy'),       F)
        np.save(os.path.join(img_out, 'sr_left.npy'), sr_left)
        np.save(os.path.join(img_out, 'sr_right.npy'),sr_right)

        # ── Accumulate ────────────────────────────────────────────────────
        all_F.append(F)
        all_labels.append(label)
        print(f'  [OK] label={label} ({"Abnormal" if label==1 else "Normal"})')

    except Exception as e:
        import traceback
        print(f'  [FAIL] {e}')
        print(traceback.format_exc())
        failed.append((image_name, str(e)))
        continue

# ── Summary ────────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('LOOP COMPLETE')
print(f'{"="*60}')
print(f'  Processed : {len(all_F)}')
print(f'  Skipped   : {len(skipped)}  (no label)')
print(f'  Failed    : {len(failed)}')
if failed:
    for name, reason in failed:
        print(f'    ✗ {name}: {reason}')

## Build and Save Dataset

In [ ]:
# ── Build arrays ───────────────────────────────────────────────────────────
F_dataset = np.array(all_F)       # (N, 21)
labels    = np.array(all_labels)  # (N,)

print(f'F_dataset shape : {F_dataset.shape}')
print(f'labels shape    : {labels.shape}')
print(f'Normal   (0)    : {(labels==0).sum()}')
print(f'Abnormal (1)    : {(labels==1).sum()}')

# ── Save ───────────────────────────────────────────────────────────────────
np.save(os.path.join(DATASET_DIR, 'F_dataset.npy'), F_dataset)
np.save(os.path.join(DATASET_DIR, 'labels.npy'),    labels)
print(f'\nSaved to {DATASET_DIR}')

# Also save a CSV for easy inspection
cols  = [f'H{i+1}'  for i in range(14)] + [f'Hu{i+1}' for i in range(7)]
df_out = pd.DataFrame(F_dataset, columns=cols)
df_out.insert(0, 'label', labels)
df_out.to_csv(os.path.join(DATASET_DIR, 'F_dataset.csv'), index=False)
print('Saved F_dataset.csv')

## FANN Classification

In [ ]:
section('FANN Classification')

# ── Minimum dataset size check ─────────────────────────────────────────────
if len(all_F) < 10:
    print(f'[WARN] Only {len(all_F)} samples — need more images for reliable classification.')
    print('       5-fold CV requires at least 5 samples per class.')
    print('       Run on full dataset before drawing conclusions.')
else:
    results = run_cross_validation(
        F_dataset = F_dataset,
        labels    = labels,
        n_splits  = 5
    )
    visualize_results(
        results,
        save_path = os.path.join(DATASET_DIR, 'classification_results.png')
    )

## Resume From Saved Results
If the loop was interrupted, reload already-processed images instead of rerunning everything.

In [ ]:
# ── Reload F vectors from per-image saved results ─────────────────────────
# Run this cell INSTEAD of the loop if you already have partial results

all_F_reload      = []
all_labels_reload = []

for image_name, label in LABEL_MAP.items():
    f_path = os.path.join(RESULTS_DIR, image_name, 'F.npy')
    if os.path.exists(f_path):
        F_loaded = np.load(f_path)
        all_F_reload.append(F_loaded)
        all_labels_reload.append(label)
        print(f'  [loaded] {image_name}')
    else:
        print(f'  [missing] {image_name} — not yet processed')

if all_F_reload:
    F_dataset_reload = np.array(all_F_reload)
    labels_reload    = np.array(all_labels_reload)
    print(f'\nReloaded {len(all_F_reload)} images')
    print(f'F_dataset shape: {F_dataset_reload.shape}')
    print(f'Normal={( labels_reload==0).sum()}, '
          f'Abnormal={(labels_reload==1).sum()}')